## 4.0 Introduction

In Lecture 3 we explored the Iris dataset with full knowledge of the species labels — we used them to color scatter plots and compute per-species centroids. But consider what we actually *used* the labels for: nothing about the centroids themselves required knowing which flower was which. We just looked at the data and computed means.

This lecture asks a natural question: **what if we didn't have the labels?** Could we still find the three clusters?

The answer is yes (mostly), and the algorithm that does it is called **k-means**. It discovers cluster structure from the data matrix $X$ alone — no label vector $\mathbf{y}$ required. This puts it in a category of methods called **unsupervised learning**: there is no ground truth to train against, only the geometry of the data.

We'll spend this lecture building k-means from the ground up: defining the objective function, deriving the update rules by hand, and running Lloyd's algorithm step by step on a small toy dataset. In Lecture 5 we'll hand the work off to Scikit-learn and apply the full pipeline to real data.

By the end of this lecture you will be able to:

- Define unsupervised learning and contrast it with supervised learning
- Define and compute $J^{\text{clust}}$, the k-means objective function
- Implement Lloyd's algorithm step by step on a small dataset
- Explain what convergence means and how to check for it

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 4.1 Supervised vs. Unsupervised Learning

Everything we have done so far in this class has been **supervised** in a loose sense: we had labels. In Lecture 3 we used the species column to color our scatter plots and compute per-species centroids. The labels told us what was "correct."

**Supervised learning** is the general setting where a model trains on labeled data — it can check its own work by comparing predictions to known outcomes.

**Unsupervised learning** is the setting where there are no labels. The algorithm must find structure in the data on its own. There is no external standard to check against.

k-means is an unsupervised method. It groups data vectors into $k$ clusters by finding centroids that minimize total distance — without ever seeing the species column. We use Iris precisely because we *do* have labels, which lets us check after the fact whether k-means found something real. In a true unsupervised problem, no such check exists.

**Question.** Think back to the prediction model from Lectures 1 and 2: $\hat{y} = w_1 x + w_2 z$. Is that supervised or unsupervised? What plays the role of the "labels" there?

## 4.2 The k-Means Objective Function

To group data vectors, we need a mathematical definition of what "good grouping" means. k-means uses **distance to the centroid** as its measure of quality.

Let $\mathbf{x}_i \in \mathbb{R}^p$ be the $i^{th}$ row vector (one observation) in the dataset, for $i = 1, \ldots, n$. Let $\mathbf{c}_1, \ldots, \mathbf{c}_k$ be $k$ **centroid vectors** — one per group. Each data vector $\mathbf{x}_i$ is assigned to exactly one group; call its centroid $\mathbf{c}_{g_i}$, where $g_i \in \{1, \ldots, k\}$ is the group index for observation $i$.

The **k-means objective function** is the mean squared distance from each data vector to its assigned centroid:

$$J^{\text{clust}} = \frac{1}{n} \sum_{i=1}^{n} \|\mathbf{x}_i - \mathbf{c}_{g_i}\|^2$$

**Question.** What is a centroid vector $\mathbf{c}_{g_i}$?

**Question.** What does $\|\mathbf{x}_i - \mathbf{c}_{g_i}\|$ measure geometrically? If every data vector sat exactly on its centroid, what would $J^{\text{clust}}$ be?

**Question.** The formula uses $\|\cdot\|^2$ rather than $\|\cdot\|$. Why might squaring the distance be mathematically convenient? (Hint: think about what squaring does to the norm formula from Lecture 2.)

**Example.** What is the distance between $\mathbf{x}_i$ and $\mathbf{c}_j$ where

$$\mathbf{x}_i = \begin{bmatrix} 5 & 5 \end{bmatrix}, \qquad \mathbf{c}_j = \begin{bmatrix} 1 & 2 \end{bmatrix}?$$

The difference vector is

$$\mathbf{x}_i - \mathbf{c}_j = \begin{bmatrix} 5 - 1 & 5 - 2 \end{bmatrix} = \begin{bmatrix} 4 & 3 \end{bmatrix}$$

and the squared distance is

$$\|\mathbf{x}_i - \mathbf{c}_j\|^2 = 4^2 + 3^2 = 16 + 9 = 25$$

so the distance is $\|\mathbf{x}_i - \mathbf{c}_j\| = \sqrt{25} = 5$.

In [ ]:
# Illustration: squared Euclidean distance between x_i and its centroid c_j
fig, ax = plt.subplots(figsize=(5, 5))

xi  = np.array([5.0, 5.0])
c_j = np.array([1.0, 2.0])

# Points
ax.scatter(*xi,  color='steelblue', s=120, zorder=4)
ax.scatter(*c_j, color='tomato',    s=350, marker='*',
           edgecolors='black', linewidth=0.8, zorder=4)

# Hypotenuse arrow: from c_j to x_i
ax.annotate('', xy=xi, xytext=c_j,
            arrowprops=dict(arrowstyle='->', color='black', lw=1.5))

# Dashed right-angle legs
ax.plot([c_j[0], xi[0]], [c_j[1], c_j[1]], color='gray', linestyle='--', lw=1.2)
ax.plot([xi[0],  xi[0]], [c_j[1], xi[1]],  color='gray', linestyle='--', lw=1.2)

# Labels
ax.annotate('$\\mathbf{x}_i$', xy=xi,  xytext=(6,  6),
            textcoords='offset points', fontsize=12, color='steelblue')
ax.annotate('$\\mathbf{c}_j$', xy=c_j, xytext=(6, -14),
            textcoords='offset points', fontsize=12, color='tomato')

# Midpoint label on hypotenuse
mid = (xi + c_j) / 2
ax.annotate('$\\|\\mathbf{x}_i - \\mathbf{c}_j\\| = 5$',
            xy=mid, xytext=(-72, 8), textcoords='offset points', fontsize=11)

# Component labels
ax.text((c_j[0] + xi[0]) / 2, c_j[1] - 0.25,
        '$x_{i1} - c_{j1} = 4$', ha='center', fontsize=9, color='gray')
ax.text(xi[0] + 0.15, (c_j[1] + xi[1]) / 2,
        '$x_{i2} - c_{j2} = 3$', ha='left', fontsize=9, color='gray')

ax.set_title('$\\|\\mathbf{x}_i - \\mathbf{c}_j\\|^2 = 4^2 + 3^2 = 25$', fontsize=11)

ax.set_xlim(0, 6.5)
ax.set_ylim(0, 6.5)
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

## 4.3 Lloyd's Algorithm

We cannot search all possible groupings of $n$ points into $k$ clusters — the number of possibilities is astronomically large even for modest $n$ and $k$. Instead, k-means uses a simple iterative procedure known as **Lloyd's algorithm** that is guaranteed to decrease $J^{\text{clust}}$ at every step.

**Lloyd's Algorithm:**

1. Choose $k$. Randomly initialize $k$ centroid vectors $\mathbf{c}_1, \ldots, \mathbf{c}_k$.
2. **Assign:** For each data vector $\mathbf{x}_i$, compute $\|\mathbf{x}_i - \mathbf{c}_j\|^2$ for all $j = 1, \ldots, k$. Assign $\mathbf{x}_i$ to the group with the nearest centroid.
3. **Update:** Recompute each centroid $\mathbf{c}_j$ as the mean of all data vectors currently assigned to group $j$.
4. Compute $J^{\text{clust}}$.
5. Repeat steps 2–4 until $J^{\text{clust}}$ stops changing — the algorithm has **converged**.

**Definition.** An algorithm has **converged** when repeating its steps produces no further change in the objective function.

**Question.** Why is step 3 (recomputing $\mathbf{c}_j$ as a mean of the group) guaranteed to decrease $J^{\text{clust}}$ or leave it unchanged? Think about what the centroid is minimizing for its assigned group — you computed this exact object in Lecture 3.

## 4.4 k-Means From Scratch

Before using `sklearn`, we'll run Lloyd's algorithm by hand on a small dataset — the same approach we took with standardization in Lecture 3: derive it manually first, then confirm with the library.

Here is the dataset:

| $A$ | $B$ |
|-----|-----|
| 1   | 1   |
| 1   | 0   |
| 0   | 2   |
| 2   | 4   |
| 3   | 5   |

We have $n = 5$ data vectors in $\mathbb{R}^2$, and we will use $k = 2$ groups. Assume both variables are already on the same scale (so no standardization is needed here).

Initial centroids: $\mathbf{c}_0 = \mathbf{x}_0 = [1, 1]$ and $\mathbf{c}_1 = \mathbf{x}_2 = [0, 2]$.

In [ ]:
# Data matrix -- each row is one observation vector x_i
X_small = np.array([[1, 1],
                    [1, 0],
                    [0, 2],
                    [2, 4],
                    [3, 5]])

n = X_small.shape[0]  # number of data vectors

# Extract individual row vectors for readability
x_0 = X_small[0, :]
x_1 = X_small[1, :]
x_2 = X_small[2, :]
x_3 = X_small[3, :]
x_4 = X_small[4, :]

In [ ]:
# A reusable helper for plotting points and centroids at any iteration.
# group_labels is a list of length n assigning each x_i to group 0 or 1.
def plot_iteration(X_small, group_labels, c_0, c_1, title):
    colors = ['steelblue', 'tomato']
    fig, ax = plt.subplots(figsize=(5, 5))
    for i, xi in enumerate(X_small):
        g = group_labels[i]
        ax.scatter(xi[0], xi[1], color=colors[g],
                   edgecolors='white', linewidth=0.5, s=80, zorder=3)
        ax.annotate(f'$\\mathbf{{x}}_{i}$', xy=(xi[0], xi[1]),
                    xytext=(5, 5), textcoords='offset points', fontsize=10)
    ax.scatter(c_0[0], c_0[1], color=colors[0], marker='*', s=350,
               edgecolors='black', linewidth=0.8, zorder=4,
               label=f'$c_0$ = {np.round(c_0, 2)}')
    ax.scatter(c_1[0], c_1[1], color=colors[1], marker='*', s=350,
               edgecolors='black', linewidth=0.8, zorder=4,
               label=f'$c_1$ = {np.round(c_1, 2)}')
    ax.set_xlabel('$A$')
    ax.set_ylabel('$B$')
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.grid(True, linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.show()

### Step 1 — Initialize: This only happens once

We initialize the three centroids by picking data points (vectors) $\mathbf{x}_0$, and $\mathbf{x}_4$.

In [ ]:
# Initialize centroids
c_0 = x_0.copy()  # centroid for group 0
c_1 = x_2.copy()  # centroid for group 1

print(f'Initial c_0: {c_0}')
print(f'Initial c_1: {c_1}')

In [ ]:
# Plot the raw data with initial centroids marked — no group assignments yet
fig, ax = plt.subplots(figsize=(5, 5))

for i, xi in enumerate(X_small):
    ax.scatter(xi[0], xi[1], color='gray',
               edgecolors='white', linewidth=0.5, s=80, zorder=3)
    ax.annotate(f'$\\mathbf{{x}}_{i}$', xy=(xi[0], xi[1]),
                xytext=(5, 5), textcoords='offset points', fontsize=10)

# Initial centroids as stars (white fill, black edge — no group color assigned yet)
for c, label in [(c_0, '$\\mathbf{c}_0$'), (c_1, '$\\mathbf{c}_1$')]:
    ax.scatter(c[0], c[1], color='white', marker='*', s=350,
               edgecolors='black', linewidth=0.8, zorder=4, label=label)

ax.set_xlabel('$A$')
ax.set_ylabel('$B$')
ax.set_title('Raw data — initial centroids, no assignments yet')
ax.legend(fontsize=9)
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

### Iteration 1

**Step 2 — Assign:** Compute the squared distance from each $\mathbf{x}_i$ to each centroid. Assign each point to its nearest centroid.

**Step 3 — Update:** Recompute each centroid as the mean of its assigned points.

**Step 4:** Compute $J_1$.

In [ ]:
# Step 2: Assign — compute squared distances and assign each x_i to its nearest centroid
# We use np.linalg.norm(v)**2 to compute ||v||^2 -- the squared Euclidean distance

print('Distances: [to c_0, to c_1] -> assigned group')
group_labels_1 = []
for i, xi in enumerate([x_0, x_1, x_2, x_3, x_4]):
    d0 = np.linalg.norm(xi - c_0)**2
    d1 = np.linalg.norm(xi - c_1)**2
    g = 0 if d0 <= d1 else 1
    group_labels_1.append(g)
    print(f'  x_{i}: [{d0:.4f}, {d1:.4f}] -> group {g}')

In [ ]:
# Step 3: Update — recompute centroids as the mean of each group's assigned vectors
# group 0: {x_0, x_1, x_2}   group 1: {x_3, x_4}

c_0 = np.mean(np.array([x_0, x_1]), axis=0)  # column-wise mean
c_1 = np.mean(np.array([x_2, x_3, x_4]), axis=0)

print(f'Updated c_0: {c_0}')
print(f'Updated c_1: {c_1}')

**Question.** `np.mean(..., axis=0)` computes the column-wise mean of a matrix — you saw this in Lecture 3 when we computed species centroids. Confirm by hand that `c_0` is the mean of $\mathbf{x}_0$, $\mathbf{x}_1$, and $\mathbf{x}_2$.

In [ ]:
# Step 4: Compute J_1
J_1 = (np.linalg.norm(x_0 - c_0)**2 +
       np.linalg.norm(x_1 - c_0)**2 +
       np.linalg.norm(x_2 - c_1)**2 +
       np.linalg.norm(x_3 - c_1)**2 +
       np.linalg.norm(x_4 - c_1)**2) / n

print(f'J_1: {J_1:.4f}')

In [ ]:
plot_iteration(X_small, group_labels_1, c_0, c_1,
               f'Iteration 1 ($J_1$ = {J_1:.4f})')

**Question.** The centroids moved from their initial positions (the data points $\mathbf{x}_0$ and $\mathbf{x}_2$) to the means of their assigned groups. Do the updated centroids look like natural "centers" of their respective clusters?

### Iteration 2

In [ ]:
# Step 2: Re-assign with updated centroids
print('Distances: [to c_0, to c_1] -> assigned group')
group_labels_2 = []
for i, xi in enumerate([x_0, x_1, x_2, x_3, x_4]):
    d0 = np.linalg.norm(xi - c_0)**2
    d1 = np.linalg.norm(xi - c_1)**2
    g = 0 if d0 <= d1 else 1
    group_labels_2.append(g)
    print(f'  x_{i}: [{d0:.4f}, {d1:.4f}] -> group {g}')

In [ ]:
# Step 3: Update centroids (if assignments haven't changed, centroids won't either)
c_0 = np.mean(np.array([x_0, x_1, x_2]), axis=0)
c_1 = np.mean(np.array([x_3, x_4]), axis=0)

# Step 4: Compute J_2
J_2 = (np.linalg.norm(x_0 - c_0)**2 +
       np.linalg.norm(x_1 - c_0)**2 +
       np.linalg.norm(x_2 - c_0)**2 +
       np.linalg.norm(x_3 - c_1)**2 +
       np.linalg.norm(x_4 - c_1)**2) / n

print(f'J_1: {J_1:.4f}')
print(f'J_2: {J_2:.4f}')
print(f'Assignments changed? {group_labels_1 != group_labels_2}')

In [ ]:
plot_iteration(X_small, group_labels_2, c_0, c_1,
               f'Iteration 2 — ($J_2$ = {J_2:.4f})')

### Iteration 3: Check for Convergence

In [ ]:
# Step 2: Re-assign with updated centroids
print('Distances: [to c_0, to c_1] -> assigned group')
group_labels_3 = []
for i, xi in enumerate([x_0, x_1, x_2, x_3, x_4]):
    d0 = np.linalg.norm(xi - c_0)**2
    d1 = np.linalg.norm(xi - c_1)**2
    g = 0 if d0 <= d1 else 1
    group_labels_3.append(g)
    print(f'  x_{i}: [{d0:.4f}, {d1:.4f}] -> group {g}')

In [ ]:
# Step 3: Update centroids (if assignments haven't changed, centroids won't either)
c_0 = np.mean(np.array([x_0, x_1, x_2]), axis=0)
c_1 = np.mean(np.array([x_3, x_4]), axis=0)

# Step 4: Compute J_2
J_3 = (np.linalg.norm(x_0 - c_0)**2 +
       np.linalg.norm(x_1 - c_0)**2 +
       np.linalg.norm(x_2 - c_0)**2 +
       np.linalg.norm(x_3 - c_1)**2 +
       np.linalg.norm(x_4 - c_1)**2) / n

print(f'J_1: {J_1:.4f}')
print(f'J_2: {J_2:.4f}')
print(f'J_3: {J_3:.4f}')
print(f'Assignments changed? {group_labels_2 != group_labels_3}')

**The algorithm has converged.** Assignments did not change between iteration 1 and iteration 2. When assignments don't change, centroids don't change, and $J^{\text{clust}}$ doesn't change — the algorithm has nothing left to do.

**Final result:** Group 0 = $\{\mathbf{x}_0, \mathbf{x}_1, \mathbf{x}_2\}$, Group 1 = $\{\mathbf{x}_3, \mathbf{x}_4\}$.

**Question.** What is the smallest $J^{\text{clust}}$ could ever be? What value of $k$ would achieve it, and why would that be a useless model?

**Looking ahead.** We now understand what k-means is doing and why it works — but running it by hand is impractical for real datasets. In Lecture 5 we'll use Scikit-learn to run k-means on the full Iris dataset, introduce PCA for visualizing high-dimensional results, and connect these ideas to the broader vocabulary of machine learning.